# Reinforcement Learning From AI Feedback (RLAIF) with Serverless customization on SageMaker AI

## Lab 4.5 - Deployment

In this notebook, we are going to deploy our fine-tuned Qwen 2.5 - 7B Instruct model to a SageMaker real-time inference endpoint.

---

## Prerequisites
### AWS Access Setup and dependencies

In [ ]:
import boto3
import os
from rich.pretty import pprint
from sagemaker.core.helper.session_helper import Session, get_execution_role

sess = Session()
sagemaker_session_bucket = None

if sagemaker_session_bucket is None and sess is not None:
    # set to default bucket if a bucket name is not given
    sagemaker_session_bucket = sess.default_bucket()

try:
    role = get_execution_role()
except ValueError:
    iam = boto3.client("iam")
    role = iam.get_role(RoleName="sagemaker_execution_role")["Role"]["Arn"]

s3_client = boto3.client("s3")
sess = Session(default_bucket=sagemaker_session_bucket)
sm_client = boto3.client("sagemaker", region_name=sess.boto_region_name)
bucket_name = sess.default_bucket()
default_prefix = sess.default_bucket_prefix

print(f"sagemaker role arn: {role}")
print(f"sagemaker bucket: {sess.default_bucket()}")
print(f"sagemaker session region: {sess.boto_region_name}")

project_prefix = "humanlike-rlaif"
base_model_jumpstart_id = "huggingface-llm-qwen2-5-7b-instruct"
base_model_shortname = "qwen25-7b"

## Prepare and create inference endpoint
### Define endpoint parameters

In [ ]:
model_package_group_name = f"{project_prefix}-{base_model_shortname}"
model_package_version = "1"

model_name = f"{project_prefix}-{base_model_shortname}"
endpoint_config_name = f"{model_package_group_name}-endpoint-config"
endpoint_name = f"{model_package_group_name}-endpoint"
ic_name = f"{model_package_group_name}-ic"

### Get fine-tuned model S3 artifact path

In [ ]:
from sagemaker.core.resources import ModelPackage, ModelPackageGroup
from sagemaker.core import s3

model_package_group = ModelPackageGroup.get(model_package_group_name)

fine_tuned_model_package_group_arn = model_package_group.model_package_group_arn
print(f"Fine-tuned Model Package Group ARN: {fine_tuned_model_package_group_arn}")

fine_tuned_model_package_arn = f"{model_package_group.model_package_group_arn.replace('model-package-group', 'model-package', 1)}/{model_package_version}"
print(f"Fine-tuned Model Package ARN: {fine_tuned_model_package_arn}")

model_package = ModelPackage.get(fine_tuned_model_package_arn)

# get the merged model artifact and deploy it
merged_model_s3_uri = (
    s3.s3_path_join(
        model_package.inference_specification.containers[
            0
        ].model_data_source.s3_data_source.s3_uri,
        "checkpoints",
        "hf_merged",
    )
    + "/"
)
print(merged_model_s3_uri)

### Create Endpoint Config

Define inference configuration

In [ ]:
instance_count = 1
instance_type = "ml.g5.2xlarge"
health_check_timeout = 700

In [ ]:
from sagemaker.core.resources import Endpoint, EndpointConfig
from sagemaker.core.shapes import ProductionVariant

print(f"Creating EndpointConfig: {endpoint_config_name}")
endpoint_config = EndpointConfig.create(
    endpoint_config_name=endpoint_config_name,
    execution_role_arn=role,
    production_variants=[
        ProductionVariant(
            variant_name="AllTraffic",
            instance_type=instance_type,
            initial_instance_count=instance_count,
            model_data_download_timeout_in_seconds=health_check_timeout,
            routing_config={"routing_strategy": "LEAST_OUTSTANDING_REQUESTS"},
        )
    ],
)

### Create Endpoint

A SageMaker Endpoint is a fully managed, always-on HTTPS API that hosts your deployed model and serves real-time inference requests.

In [ ]:
print(f"Creating Endpoint: {endpoint_name}")
endpoint = Endpoint.create(
    endpoint_name=endpoint_name, endpoint_config_name=endpoint_config_name
)
endpoint.wait_for_status("InService")
print(f"Endpoint {endpoint_name} is InService")

### Create Model from Model Package

Get the image URI

In [ ]:
import json

In [ ]:
region = sess.boto_region_name
CONTAINER_VERSION = "0.36.0-lmi18.0.0-cu128"

inference_image = (
    f"763104351884.dkr.ecr.{region}.amazonaws.com/djl-inference:{CONTAINER_VERSION}"
)

In [ ]:
env = {
    "HF_MODEL_ID": "/opt/ml/model",  # path to where sagemaker stores the model
    "OPTION_TRUST_REMOTE_CODE": "true",
    "OPTION_MODEL_LOADING_TIMEOUT": "3600",
    "OPTION_TENSOR_PARALLEL_DEGREE": "max",
    "SERVING_FAIL_FAST": "true",
    "OPTION_ROLLING_BATCH": "disable",
    "OPTION_ASYNC_MODE": "true",
    "OPTION_ENTRYPOINT": "djl_python.lmi_vllm.vllm_async_service",
    "OPTION_DTYPE": "bf16",
    "OPTION_QUANTIZE": "fp8",
    "OPTION_MAX_MODEL_LEN": json.dumps(1024 * 32)
}

In [ ]:
from sagemaker.core.resources import Model
from sagemaker.core.resources import TrainingJob
from sagemaker.core.shapes import (
    ContainerDefinition,
    ModelDataSource,
    S3ModelDataSource,
)

fine_tuned_model = Model.create(
    model_name=model_name,
    primary_container=ContainerDefinition(
        image=inference_image,
        model_data_source=ModelDataSource(
            s3_data_source=S3ModelDataSource(
                s3_uri=merged_model_s3_uri,
                s3_data_type="S3Prefix",
                compression_type="None",
            )
        ),
        environment=env,
    ),
    execution_role_arn=role,
)

pprint(fine_tuned_model)

### Create Inference Component

In [ ]:
from sagemaker.core.resources import InferenceComponent
from sagemaker.core.shapes import (
    InferenceComponentSpecification,
    InferenceComponentComputeResourceRequirements,
    InferenceComponentRuntimeConfig,
)

# Step 3: Create InferenceComponent
inference_component = InferenceComponent.create(
    inference_component_name=ic_name,
    endpoint_name=endpoint_name,
    variant_name="AllTraffic",
    specification=InferenceComponentSpecification(
        model_name=model_name,
        compute_resource_requirements=InferenceComponentComputeResourceRequirements(
            min_memory_required_in_mb=10240,
            number_of_accelerator_devices_required=1,
        ),
    ),
    runtime_config=InferenceComponentRuntimeConfig(copy_count=1),
    region=region,
)

print(f"InferenceComponent created: {inference_component.inference_component_name}")
print(f"Endpoint ARN: {endpoint.endpoint_arn}")
inference_component.wait_for_status("InService")
print(f"Endpoint {ic_name} is InService")

## Test the inference endpoint
### Test endpoint

In [ ]:
import io
import json
import boto3

In [ ]:
sagemaker_client = boto3.client(service_name="sagemaker-runtime")

### Iterator class for streaming inference

Utility class to parse streaming responses

In [ ]:
class LineIterator:
    def __init__(self, stream):
        self.byte_iterator = iter(stream)
        self.buffer = io.BytesIO()
        self.read_pos = 0

    def __iter__(self):
        return self

    def __next__(self):
        while True:
            self.buffer.seek(self.read_pos)
            line = self.buffer.readline()

            if line and line[-1] == ord("\n"):
                self.read_pos += len(line)
                return line[:-1]

            try:
                chunk = next(self.byte_iterator)
            except StopIteration:
                if self.read_pos < self.buffer.getbuffer().nbytes:
                    continue
                raise

            if "PayloadPart" not in chunk:
                continue

            self.buffer.seek(0, io.SEEK_END)
            self.buffer.write(chunk["PayloadPart"]["Bytes"])

Utility function to parse model answer

In [ ]:
def parse_streaming_response(line_str):
    """Parse a single streaming response line and return content if found."""
    if not line_str.strip() or line_str.strip() == "data: [DONE]":
        return None

    if line_str.startswith("data: "):
        line_str = line_str[6:]

    try:
        data = json.loads(line_str)
        if "choices" in data:
            for choice in data["choices"]:
                if "delta" in choice and "content" in choice["delta"]:
                    return choice["delta"]["content"]
    except json.JSONDecodeError:
        pass

    return None

***

In [ ]:
prompt = """
What's your honest take on pineapple on pizza?
"""

In [ ]:
request_body = {
    "model_name": ic_name,
    "messages": [
        {
            "role": "user",
            "content": [
                {"type": "text", "text": prompt},
            ],
        }
    ],
    "max_tokens": 4096,
    "temperature": 0.3,
    "top_p": 0.9,
    "stop": ["<|im_end|>"],
    "stream": True,
}

response = sagemaker_client.invoke_endpoint_with_response_stream(
    EndpointName=endpoint_name,
    InferenceComponentName=ic_name,
    Body=json.dumps(request_body),
    ContentType="application/json",
)

generated_text = ""

for line in LineIterator(response["Body"]):
    if line:
        content = parse_streaming_response(line.decode("utf-8"))
        if content:
            generated_text += content
            print(content, end="", flush=True)

## Clean up

In [ ]:
from sagemaker.core.resources import InferenceComponent

# Delete inference component
InferenceComponent.get(inference_component_name=ic_name).delete()

In [ ]:
from sagemaker.core.resources import Model

# Delete model
Model.get(model_name=model_name).delete()

In [ ]:
from sagemaker.core.resources import Endpoint

# Delete endpoint (optional - if you want to remove the endpoint too)
Endpoint.get(endpoint_name=endpoint_name).delete()

In [ ]:
from sagemaker.core.resources import EndpointConfig

# Delete endpoint config (optional)
EndpointConfig.get(endpoint_config_name=endpoint_config_name).delete()